# Troubleshooting sufficiency — model comparison

Goal: predict `1` (insufficient information) vs `0` (sufficient) for tech-support questions.
Metric: **F1 of class 1** (the public-leaderboard scores 6/13, 16/27, 9/17 fit binary F1 on ~25 public rows).

Every model is scored with the same 5-fold stratified CV repeated 5 times (`feats.cv_eval`), with the
decision threshold chosen inside each training part (inner CV), never on the held-out fold.

Reference points:
* predicting `1` for everything gives F1 ≈ 0.51 (22 of 64 questions are class 1) — any model must beat this;
* run1 (mpnet embeddings + LR, threshold 0.5) — CV 0.454, public 0.46153.

Feature families:
1. **handcrafted** regex cues (`feats.handcrafted`): length, versions/OS/device named, error mentioned/quoted, tried steps, attachments…
2. **sentence embeddings** of the full text from cached encoders (mpnet, BERT-base);
3. **LLM checklist** (`llm_judge.py`): a local LLM answers Yes/No questions ("enough information to start troubleshooting?",
   "product and version named?", …); the soft P(Yes) values are the features. Prompts are saved in `features/llm_prompts_*.json`.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, random, torch
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, roc_auc_score
from feats import SEED, handcrafted, cv_eval, best_threshold, oof_proba

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
train, test = pd.read_csv("data/train.csv"), pd.read_csv("data/test.csv")
y, n = train.sufficiency.values, len(train)
all_text = np.concatenate([train.text.values, test.text.values])
print(train.shape, test.shape, "class-1 share", y.mean().round(3))
print("F1 of predicting all 1:", round(f1_score(y, np.ones_like(y)), 3))

(64, 3) (50, 2) class-1 share 0.344
F1 of predicting all 1: 0.512


## Features

In [2]:
H = handcrafted(all_text)
E = {k: np.load(f"features/emb_{k}.npy") for k in ["mpnet", "bert"]}      # computed with feats.embed (see exp scripts)

L = {}
for tag in ["q25", "q3"]:
    try:
        d = pd.read_csv(f"features/llm_{tag}.csv").set_index("id").loc[np.concatenate([train.id, test.id])]
        L[tag] = d[[c for c in d.columns if not c.endswith("_mass")]]
        print(tag, "mean Yes/No mass", d.filter(like="_mass").mean().round(2).to_dict())
    except FileNotFoundError:
        print("missing", tag)

# How well does each single LLM answer separate the classes on train? (AUC 0.5 = no signal)
auc = {f"{tag}:{c}": roc_auc_score(y, L[tag][c].values[:n]) for tag in L for c in L[tag]}
pd.Series(auc).sort_values().round(3)

q25 mean Yes/No mass {'sufficient_mass': 1.0, 'need_more_mass': 1.0, 'product_version_mass': 1.0, 'clear_symptom_mass': 1.0, 'error_text_mass': 1.0, 'tried_steps_mass': 1.0, 'missing_attachment_mass': 1.0, 'vague_mass': 1.0}
missing q3


q25:product_version       0.310
q25:error_text            0.332
q25:sufficient            0.390
q25:clear_symptom         0.400
q25:tried_steps           0.437
q25:vague                 0.671
q25:need_more             0.685
q25:missing_attachment    0.687
dtype: float64

## Cross-validated comparison

In [3]:
def LR(C, scale=True):
    m = LogisticRegression(C=C, class_weight="balanced", max_iter=5000, random_state=SEED)
    return make_pipeline(StandardScaler(), m) if scale else m

results, oofs = {}, {}
def run(name, model, X):
    r = cv_eval(model, X[:n], y)
    results[name] = {k: v for k, v in r.items() if k != "oof"}
    oofs[name] = r["oof"]
    print(f"{name:45s} F1 tuned {r['f1_tuned']:.3f} ± {r['f1_tuned_std']:.3f}   F1@0.5 {r['f1@0.5']:.3f}")

Hv = H.values.astype(float)
run("handcrafted LR C=0.03", LR(0.03), Hv)
run("mpnet LR C=1", LR(1, scale=False), E["mpnet"])
run("bert LR C=1", LR(1, scale=False), E["bert"])
run("bert SVM C=10", SVC(C=10, probability=True, class_weight="balanced", random_state=SEED), E["bert"])

for tag in L:
    Lv = L[tag].values
    run(f"LLM {tag}: 'sufficient' only, LR", LR(1), Lv[:, [0]])
    for C in (0.03, 0.3):
        run(f"LLM {tag}: checklist LR C={C}", LR(C), Lv)
    run(f"LLM {tag}: checklist + handcrafted LR C=0.03", LR(0.03), np.hstack([Lv, Hv]))
if len(L) == 2:
    both = np.hstack([L["q25"].values, L["q3"].values])
    run("LLM q25+q3 checklist LR C=0.03", LR(0.03), both)

handcrafted LR C=0.03                         F1 tuned 0.487 ± 0.078   F1@0.5 0.390


mpnet LR C=1                                  F1 tuned 0.449 ± 0.137   F1@0.5 0.426


bert LR C=1                                   F1 tuned 0.496 ± 0.086   F1@0.5 0.501


bert SVM C=10                                 F1 tuned 0.510 ± 0.060   F1@0.5 0.048


LLM q25: 'sufficient' only, LR                F1 tuned 0.458 ± 0.133   F1@0.5 0.441


LLM q25: checklist LR C=0.03                  F1 tuned 0.589 ± 0.136   F1@0.5 0.611


LLM q25: checklist LR C=0.3                   F1 tuned 0.536 ± 0.143   F1@0.5 0.575


LLM q25: checklist + handcrafted LR C=0.03    F1 tuned 0.473 ± 0.120   F1@0.5 0.496


### Small neural network

A one-hidden-layer MLP (16 units, strong L2) on the LLM checklist + handcrafted features.

In [4]:
best_tag = max(L, key=lambda t: results[f"LLM {t}: checklist LR C=0.03"]["f1_tuned"]) if L else None
if best_tag:
    X_nn = np.hstack([L[best_tag].values, Hv])
    run(f"MLP 16 on LLM {best_tag} + handcrafted",
        make_pipeline(StandardScaler(), MLPClassifier((16,), alpha=1.0, max_iter=3000, random_state=SEED)), X_nn)

MLP 16 on LLM q25 + handcrafted               F1 tuned 0.429 ± 0.114   F1@0.5 0.350


## Results

In [5]:
res = pd.DataFrame(results).T.sort_values("f1_tuned", ascending=False)
res.round(3)

,f1@0.5,f1@0.5_std,f1_tuned,f1_tuned_std
LLM q25: checklist LR C=0.03,0.611,0.112,0.589,0.136
LLM q25: checklist LR C=0.3,0.575,0.142,0.536,0.143
bert SVM C=10,0.048,0.112,0.510,0.060
bert LR C=1,0.501,0.167,0.496,0.086
handcrafted LR C=0.03,0.390,0.192,0.487,0.078
LLM q25: checklist + handcrafted LR C=0.03,0.496,0.127,0.473,0.120
"LLM q25: 'sufficient' only, LR",0.441,0.113,0.458,0.133
mpnet LR C=1,0.426,0.221,0.449,0.137
MLP 16 on LLM q25 + handcrafted,0.350,0.151,0.429,0.114


## Submission

Fit the chosen model on all 64 questions, choose the threshold from out-of-fold probabilities on the
training set (same rule as inside CV), and predict the 50 test questions.

In [6]:
BEST = res.index[0]          # <- best by CV; change by hand to submit another row
print("chosen:", BEST, results[BEST])

chosen: LLM q25: checklist LR C=0.03 {'f1@0.5': np.float64(0.6107359307359307), 'f1@0.5_std': np.float64(0.1119101675836512), 'f1_tuned': np.float64(0.588976134976135), 'f1_tuned_std': np.float64(0.13608279476540644)}
